# Gold Sector Overview Mart

**Purpose**: Cross-sector market summary for executive dashboards.

**Target Table**: `workspace.gold.gold_sector_overview`

**Grain**: Date + sector combinations with comprehensive market metrics

**Refresh Strategy**: Full refresh (CREATE OR REPLACE) with metadata logging

**Parameters**:
- `lookback_days`: Number of days of historical data (default: 365)
- `force_full_refresh`: Boolean flag to force table recreation

**Key Metrics**:
- Market size (jobs, companies, roles, locations)
- Growth indicators (new jobs, closed jobs)
- Skill diversity
- Salary ranges (median, p25, p75)

**Data Sources**:
- `workspace.warehouse.fact_job_postings`
- `workspace.warehouse.fact_salary`
- `workspace.warehouse.bridge_job_skill`

**Metadata Logging**:
- Tracks run history in `workspace.metadata.gold_sector_overview_refresh_log`
- Captures: rows processed, unique dates/sectors, processing time, status

In [0]:
# Configuration
target_table = "workspace.gold.gold_sector_overview"
metadata_table = "workspace.metadata.gold_sector_overview_refresh_log"

# Parameters
lookback_days = 365  # How many days of history to include
force_full_refresh = False  # Set to True to drop and recreate table

In [0]:
import uuid
from datetime import datetime

# Generate unique run ID
run_id = str(uuid.uuid4())
run_timestamp = datetime.now()
status = "RUNNING"
error_message = None

print(f"Run ID: {run_id}")
print(f"Run Timestamp: {run_timestamp}")
print(f"Lookback Days: {lookback_days}")
print(f"Force Full Refresh: {force_full_refresh}")

In [0]:
%sql
-- Create table with all columns if it doesn't exist
CREATE TABLE IF NOT EXISTS workspace.metadata.gold_sector_overview_refresh_log (
  run_id STRING NOT NULL,
  run_timestamp TIMESTAMP NOT NULL,
  rows_inserted BIGINT,
  status STRING NOT NULL,
  lookback_days INT,
  rows_processed BIGINT,
  unique_dates INT,
  unique_sectors INT,
  force_full_refresh BOOLEAN,
  processing_time_seconds DECIMAL(10,2),
  error_message STRING
)
USING DELTA
COMMENT 'Audit log for gold_sector_overview refresh operations';

In [0]:
# Optionally drop table for full refresh
if force_full_refresh:
    print(f"Force full refresh enabled - dropping table {target_table}")
    spark.sql(f"DROP TABLE IF EXISTS {target_table}")
    print("✓ Table dropped")
else:
    print("Proceeding with CREATE OR REPLACE (preserves table properties)")

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_sector_overview (
  sector_date_sk INT NOT NULL COMMENT 'Date key YYYYMMDD',
  sector_sk BIGINT NOT NULL COMMENT 'FK to dim_sector',
  active_jobs INT COMMENT 'Active jobs count',
  new_jobs INT COMMENT 'New jobs posted',
  closed_jobs INT COMMENT 'Jobs closed',
  active_companies INT COMMENT 'Distinct companies',
  active_roles INT COMMENT 'Distinct roles',
  active_locations INT COMMENT 'Distinct locations',
  unique_skills_demanded INT COMMENT 'Distinct skills',
  jobs_per_company DECIMAL(10,2) COMMENT 'Market concentration',
  median_salary_max DECIMAL(18,2) COMMENT 'Median max salary',
  salary_observations INT COMMENT 'Salary data points'
)
USING DELTA
COMMENT 'Cross-sector market overview for executive dashboards'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
from datetime import datetime, timedelta

# Calculate cutoff date
cutoff_date = int((datetime.now() - timedelta(days=lookback_days)).strftime("%Y%m%d"))

print(f"Building sector overview (lookback: {lookback_days} days, cutoff: {cutoff_date})...")

# Insert aggregated data
insert_sql = f"""
INSERT OVERWRITE TABLE {target_table}

WITH job_metrics AS (
  SELECT
    posting_date_sk AS sector_date_sk,
    sector_sk,
    COUNT(CASE WHEN active_flag = TRUE THEN 1 END) AS active_jobs,
    COUNT(CASE WHEN is_new_job = TRUE THEN 1 END) AS new_jobs,
    COUNT(CASE WHEN is_soft_delete = TRUE THEN 1 END) AS closed_jobs,
    COUNT(DISTINCT CASE WHEN active_flag = TRUE THEN company_sk END) AS active_companies,
    COUNT(DISTINCT CASE WHEN active_flag = TRUE THEN role_sk END) AS active_roles,
    COUNT(DISTINCT CASE WHEN active_flag = TRUE THEN location_sk END) AS active_locations
  FROM workspace.warehouse.fact_job_postings
  WHERE posting_date_sk >= {cutoff_date}
  GROUP BY posting_date_sk, sector_sk
),

skill_metrics AS (
  SELECT
    f.posting_date_sk,
    f.sector_sk,
    COUNT(DISTINCT bjs.skill_sk) AS unique_skills_demanded
  FROM workspace.warehouse.fact_job_postings f
  JOIN workspace.warehouse.bridge_job_skill bjs ON f.job_sk = bjs.job_sk
  WHERE f.posting_date_sk >= {cutoff_date}
    AND f.active_flag = TRUE
    AND bjs.is_current = TRUE
  GROUP BY f.posting_date_sk, f.sector_sk
),

salary_metrics AS (
  SELECT
    fs.observation_date_sk,
    j.sector_sk,  -- Get sector from dim_job, not fact_salary
    PERCENTILE(fs.salary_max, 0.50) AS median_salary_max,
    COUNT(*) AS salary_observations
  FROM workspace.warehouse.fact_salary fs
  INNER JOIN workspace.warehouse.dim_job j ON fs.job_sk = j.job_sk AND j.is_current = TRUE
  WHERE fs.observation_date_sk >= {cutoff_date}
    AND fs.salary_period = 'ANNUAL'
    AND fs.salary_currency = 'USD'
    AND fs.salary_max > 0
  GROUP BY fs.observation_date_sk, j.sector_sk
)

SELECT
  jm.sector_date_sk,
  jm.sector_sk,
  jm.active_jobs,
  jm.new_jobs,
  jm.closed_jobs,
  jm.active_companies,
  jm.active_roles,
  jm.active_locations,
  COALESCE(sm.unique_skills_demanded, 0) AS unique_skills_demanded,
  CASE 
    WHEN jm.active_companies > 0 
    THEN CAST(jm.active_jobs AS DECIMAL(10,2)) / jm.active_companies
    ELSE NULL 
  END AS jobs_per_company,
  CAST(COALESCE(sal.median_salary_max, 0) AS DECIMAL(18,2)) AS median_salary_max,
  COALESCE(sal.salary_observations, 0) AS salary_observations
FROM job_metrics jm
LEFT JOIN skill_metrics sm 
  ON jm.sector_date_sk = sm.posting_date_sk 
  AND jm.sector_sk = sm.sector_sk
LEFT JOIN salary_metrics sal 
  ON jm.sector_date_sk = sal.observation_date_sk 
  AND jm.sector_sk = sal.sector_sk
"""

spark.sql(insert_sql)
print("✓ Data loaded")

In [0]:
import time

start_time = time.time()

try:
    # Get summary metrics from the gold table
    metrics = spark.sql(f"""
        SELECT 
            COUNT(*) as rows_processed,
            COUNT(DISTINCT sector_date_sk) as unique_dates,
            COUNT(DISTINCT sector_sk) as unique_sectors
        FROM {target_table}
    """).collect()[0]
    
    rows_processed = metrics['rows_processed']
    unique_dates = metrics['unique_dates']
    unique_sectors = metrics['unique_sectors']
    processing_time = time.time() - start_time
    status = "SUCCESS"
    
    print(f"✓ Processed {rows_processed:,} rows")
    print(f"✓ Unique dates: {unique_dates}")
    print(f"✓ Unique sectors: {unique_sectors}")
    print(f"✓ Processing time: {processing_time:.2f}s")
    
except Exception as e:
    status = "FAILED"
    error_message = str(e)
    processing_time = time.time() - start_time
    rows_processed = 0
    unique_dates = 0
    unique_sectors = 0
    print(f"✗ Error: {error_message}")
    raise

finally:
    # Write metadata log
    spark.sql(f"""
        INSERT INTO {metadata_table} (
            run_id,
            run_timestamp,
            rows_inserted,
            status,
            lookback_days,
            rows_processed,
            unique_dates,
            unique_sectors,
            force_full_refresh,
            processing_time_seconds,
            error_message
        )
        VALUES (
            '{run_id}',
            TIMESTAMP'{run_timestamp}',
            {rows_processed},
            '{status}',
            {lookback_days},
            {rows_processed},
            {unique_dates},
            {unique_sectors},
            {force_full_refresh},
            {processing_time:.2f},
            {'NULL' if error_message is None else f"'{error_message}'"}
        )
    """)

In [0]:
%sql
-- Overall summary statistics
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT sector_date_sk) AS unique_dates,
  COUNT(DISTINCT sector_sk) AS unique_sectors,
  MIN(sector_date_sk) AS earliest_date,
  MAX(sector_date_sk) AS latest_date,
  SUM(active_jobs) AS total_active_jobs,
  SUM(new_jobs) AS total_new_jobs,
  SUM(closed_jobs) AS total_closed_jobs,
  AVG(jobs_per_company) AS avg_jobs_per_company,
  AVG(median_salary_max) AS avg_median_salary
FROM workspace.gold.gold_sector_overview;

In [0]:
%sql
-- Sector comparison
SELECT
  ds.sector_name,
  COUNT(DISTINCT gso.sector_date_sk) AS days_active,
  SUM(gso.active_jobs) AS total_active_jobs,
  SUM(gso.new_jobs) AS total_new_jobs,
  ROUND(AVG(gso.jobs_per_company), 2) AS avg_jobs_per_company,
  ROUND(AVG(gso.median_salary_max), 0) AS avg_median_salary,
  AVG(gso.unique_skills_demanded) AS avg_unique_skills
FROM workspace.gold.gold_sector_overview gso
LEFT JOIN workspace.warehouse.dim_sector ds ON gso.sector_sk = ds.sector_sk
GROUP BY ds.sector_name
ORDER BY total_active_jobs DESC;

In [0]:
%sql
-- Check for data quality issues
SELECT
  'Null sector_date_sk' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_sector_overview
WHERE sector_date_sk IS NULL

UNION ALL

SELECT
  'Null sector_sk' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_sector_overview
WHERE sector_sk IS NULL

UNION ALL

SELECT
  'Negative active_jobs' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_sector_overview
WHERE active_jobs < 0

UNION ALL

SELECT
  'Invalid median_salary_max' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_sector_overview
WHERE median_salary_max < 0

ORDER BY issue_count DESC;

In [0]:
%sql
-- View last 10 refresh runs
SELECT 
  run_id,
  run_timestamp,
  lookback_days,
  rows_processed,
  unique_dates,
  unique_sectors,
  processing_time_seconds,
  status,
  error_message
FROM workspace.metadata.gold_sector_overview_refresh_log
ORDER BY run_timestamp DESC
LIMIT 10;